# Phase 1: Baseline Training - ResNet50

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, os.getcwd())

# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

# Project imports
from config import *
from dataloader import load_and_split_data, create_dataloaders, get_class_counts, get_pos_weights
from models.resnet import ResNet50Classifier
from losses import get_loss_function
from trainer import train_model, validate
from utils import set_seed

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {DEVICE}")

In [ ]:
# Set seed for reproducibility
set_seed(RANDOM_SEED)

# Load and split data (patient-level split)
train_df, val_df = load_and_split_data()

# Get class counts for weighted loss
class_counts = get_class_counts(train_df)
print(f"\nClass counts (sorted):")
for name, count in sorted(zip(CLASS_NAMES, class_counts), key=lambda x: -x[1]):
    print(f"  {name}: {int(count)}")

In [ ]:
# Visualize class distribution
fig, ax = plt.subplots(figsize=(14, 6))
sorted_idx = np.argsort(class_counts)[::-1]
ax.bar(range(NUM_CLASSES), class_counts[sorted_idx])
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels([CLASS_NAMES[i] for i in sorted_idx], rotation=45, ha='right')
ax.set_ylabel('Count')
ax.set_title('Class Distribution (Long-Tail)')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print(f"\nImbalance ratio: {class_counts.max() / class_counts.min():.1f}:1")

## Create DataLoaders

In [ ]:
# Create dataloaders
train_loader, val_loader = create_dataloaders(train_df, val_df)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Batch size: {BATCH_SIZE}")

In [ ]:
# Test dataloader - verify a batch
images, labels = next(iter(val_loader))
print(f"Image batch shape: {images.shape}")
print(f"Label batch shape: {labels.shape}")
print(f"Label sum per sample (avg): {labels.sum(dim=1).mean():.2f}")

# Show sample images
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i in range(4):
    img = images[i].permute(1, 2, 0).numpy()
    img = (img * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]  # Denormalize
    img = np.clip(img, 0, 1)
    axes[i].imshow(img[:,:,0])
    active_labels = [CLASS_NAMES[j] for j in range(NUM_CLASSES) if labels[i, j] == 1]
    axes[i].set_title(f"{active_labels[:2]}..." if len(active_labels) > 2 else str(active_labels))
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## Initialize Model

In [ ]:
# Initialize ResNet50 model
model = ResNet50Classifier(num_classes=NUM_CLASSES, pretrained=True, dropout=0.5)
model = model.to(DEVICE)

print(f"Model: ResNet50")
print(f"Total parameters: {model.get_total_params():,}")
print(f"Trainable parameters: {model.get_trainable_params():,}")

In [ ]:
# Setup loss, optimizer, scheduler
# Using weighted BCE loss for baseline
pos_weight = get_pos_weights(train_df)
criterion = get_loss_function("bce", pos_weight=pos_weight)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

print(f"Loss: BCEWithLogitsLoss (weighted)")
print(f"Optimizer: AdamW (lr={LEARNING_RATE})")
print(f"Scheduler: CosineAnnealingLR")

## Training

In [ ]:
# Train the model
history, best_mAP = train_model(
    model=model,
    train_loader=val_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=NUM_EPOCHS,
    early_stopping_patience=7,
    model_name="resnet50_baseline"
)

print(f"\nBest Validation mAP: {best_mAP:.4f}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss')
axes[0].legend()

# mAP
axes[1].plot(history['mAP'])
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('mAP')
axes[1].set_title(f'Validation mAP (Best: {max(history["mAP"]):.4f})')

# mAUC
axes[2].plot(history['mAUC'])
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('mAUC')
axes[2].set_title(f'Validation mAUC (Best: {max(history["mAUC"]):.4f})')

plt.tight_layout()
plt.show()

## Evaluate Best Model

In [ ]:
# Load best model and evaluate
from trainer import load_checkpoint

best_model = ResNet50Classifier(num_classes=NUM_CLASSES, pretrained=False)
load_checkpoint(best_model, None, os.path.join(CHECKPOINT_DIR, "resnet50_baseline_best.pth"))
best_model = best_model.to(DEVICE)

# Final validation
val_metrics = validate(best_model, val_loader, criterion)
print(f"\nFinal Validation Metrics:")
print(f"  mAP: {val_metrics['mAP']:.4f}")
print(f"  mAUC: {val_metrics['mAUC']:.4f}")
print(f"  mF1: {val_metrics['mF1']:.4f}")

# Per-class AP
print(f"\nPer-Class AP:")
for name, ap in sorted(zip(CLASS_NAMES, val_metrics['per_class_ap']), key=lambda x: -x[1]):
    print(f"  {name}: {ap:.4f}")

## Generate Test Predictions

In [ ]:
# Create test predictions
from dataloader import create_test_dataloader
from trainer import predict
from utils import create_submission, validate_submission

test_loader, test_df = create_test_dataloader()
print(f"Test samples: {len(test_df)}")

# Generate predictions
image_ids, predictions = predict(best_model, test_loader)

# Create submission
submission_df = create_submission(image_ids, predictions, "submission_resnet50_baseline.csv")

# Validate format
validate_submission(os.path.join(OUTPUT_DIR, "submission_resnet50_baseline.csv"))

In [ ]:
print("Phase 1 Complete!")
print(f"Best mAP: {best_mAP:.4f}")
print(f"Submission saved to: {os.path.join(OUTPUT_DIR, 'submission_resnet50_baseline.csv')}")